<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/solutions/09_generative_solutions.ipynb)

# 第五部分 · 练习参考答案

⚠️ **先自己做，再看答案。** 写法不唯一，能得到同样结果的都对。

生成模型的结果和随机种子关系很大，你的数字和这里差一些是正常的，但**趋势**应该一致。

整个 notebook 运行大约 12 分钟，其中两次 VAE 训练占一半。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models/solutions"))
    get_ipython().run_line_magic("pip", "install -q rdkit selfies")


---
## 08a

先把 08a 的准备工作做一遍（代码和 notebook 里完全相同）：

In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
print("PyTorch 版本:", torch.__version__)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 cep.csv 即可
DATA_DIR = "../data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv（如果要用预训练模型，把 cep_rnn.pt 也一起上传）")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print("读取成功！一共", len(df), "个分子")
df.head()

In [ ]:
TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")

def tokenize(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)
    assert "".join(tokens) == smiles, f"分词结果拼不回原串: {smiles}"   # 一定要检查！
    return tokens

print(tokenize("c1ccsc1"))
print(tokenize("[nH]1c(cc2c3nsnc3c3c4ccccc4oc3c12)-c1ccco1"))
print(tokenize("CC(=O)Oc1ccccc1C(=O)O"))     # 阿司匹林，带支链和官能团

In [ ]:
token_lists = [tokenize(s) for s in df["smiles"]]
counter = collections.Counter(t for ts in token_lists for t in ts)
print("一共", len(counter), "种 token")
pd.Series(counter).sort_values(ascending=False).to_frame("count")

In [ ]:
vocab = ["<pad>", "^", "$"] + sorted(counter)
stoi = {t: i for i, t in enumerate(vocab)}            # token -> 编号
max_len = max(len(ts) for ts in token_lists) + 2      # +2 是给 ^ 和 $ 留位置
print("词表大小:", len(vocab), " 最长序列:", max_len)
print(vocab)

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, :len(ids)] = torch.tensor(ids)

print("\n第一个分子:", df["smiles"][0])
print("编码后（0 是补齐）:", data[0][:20].tolist(), "...")

In [ ]:
class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)   # 每个 token -> 一个向量
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)           # 隐状态 -> 每个 token 的分数

    def forward(self, x, state=None):
        """x: (batch, 长度) 的 token 编号。返回每个位置上"下一个 token"的分数。"""
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

In [ ]:
@torch.no_grad()
def sample(model, n=500, temperature=1.0, max_steps=max_len):
    """从模型里采样 n 个 SMILES 字符串。"""
    model.eval()
    x = torch.full((n, 1), stoi["^"])             # 所有序列都从 ^ 开始
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_steps):
        logits, state = model(x, state)            # 只喂最新的一个 token，state 记住了前面的内容
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)            # 按概率抽一个 token（不是取最大的！）
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi["$"])
        if finished.all():
            break
    model.train()

    smiles_list = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        smiles_list.append("".join(tokens))
    return smiles_list

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def evaluate(samples, name="model"):
    """生成模型的三个标准指标：有效性、唯一性、新颖性。"""
    mols = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in mols if m is not None and m.GetNumAtoms() > 0]
    unique = set(valid)
    novel = unique - canonical_train
    row = {
        "valid": len(valid) / len(samples),                 # 能被 RDKit 读懂的比例
        "unique": len(unique) / max(len(valid), 1),         # 有效分子里不重复的比例
        "novel": len(novel) / max(len(unique), 1),          # 不重复分子里训练集没有的比例
    }
    print(f"{name:22s} valid {row['valid']:.3f}  unique {row['unique']:.3f}  novel {row['novel']:.3f}")
    return row

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]

CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
rnn = SmilesRNN(len(vocab))
rnn.load_state_dict(torch.load(CKPT, map_location="cpu"))
rnn.eval()
print("已加载预训练模型")

### 08a-1

In [ ]:
samples = sample(rnn, 2000)

syntax_errors, chemistry_errors, problem_types = [], [], collections.Counter()
for s in samples:
    if Chem.MolFromSmiles(s) is not None:
        continue                                        # 有效，跳过
    raw = Chem.MolFromSmiles(s, sanitize=False)         # 不做化学检查，只做语法解析
    if raw is None:
        syntax_errors.append(s)                         # 连解析都失败 = 语法错
    else:
        chemistry_errors.append(s)
        for problem in Chem.DetectChemistryProblems(raw):
            problem_types[problem.GetType()] += 1

n_invalid = len(syntax_errors) + len(chemistry_errors)
print(f"2000 个样本里 {n_invalid} 个无效（{n_invalid / 2000:.1%}）")
print(f"  语法错（括号/环号不配对）: {len(syntax_errors)}  ({len(syntax_errors) / max(n_invalid, 1):.1%})")
print(f"  化学错（能解析但化合价等不对）: {len(chemistry_errors)}  ({len(chemistry_errors) / max(n_invalid, 1):.1%})")
print("\n化学错误的类型:", dict(problem_types))
print("\n语法错的例子:")
for s in syntax_errors[:3]:
    print("  ", s)
print("化学错的例子:")
for s in chemistry_errors[:3]:
    print("  ", s)

📘 2000 个样本里只有几十个无效（百分之几），而且**两类错误几乎各占一半**：

- **语法错**：括号没配对、环号落单。看例子就很清楚：`...sc4c3[SiH2]C=Cc3c2=C1` 里出现了落单的 `4`，模型"开了一个环却忘了关"。
- **化学错**：绝大多数是 `KekulizeException`，少数是 `AtomValenceException`。

`KekulizeException` 是什么意思？模型写出了一圈小写字母（芳香原子），但 RDKit 没办法给它分配交替的单双键——也就是说，**这个"芳香环"在化学上不可能存在**。

🤔 这说明模型学到了什么、没学到什么？

📘 它学会了 SMILES 的**表面统计规律**（小写字母成串出现、数字成对出现），但没有学会**休克尔规则**。它只是在模仿字符串的样子，不理解背后的化学——这正是 08a 第 1 节那句话的具体体现。

### 08a-2

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
reference_idx = np.random.RandomState(0).choice(len(mols), 3000, replace=False)
reference_fps = [fp_gen.GetFingerprint(mols[i]) for i in reference_idx]

generated = [Chem.MolFromSmiles(s) for s in sample(rnn, 1200)]
generated = [m for m in generated if m is not None and m.GetNumAtoms() > 0][:500]
gen_sim = np.array([max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(m), reference_fps))
                    for m in generated])

inside_idx = np.random.RandomState(1).choice(len(mols), 500, replace=False)
inside_sim = np.array([sorted(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mols[i]), reference_fps))[-2]
                       for i in inside_idx])

fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(0.2, 1.0, 40)
ax.hist(inside_sim, bins=bins, density=True, alpha=0.5, label="CEP molecule vs CEP")
ax.hist(gen_sim, bins=bins, density=True, histtype="step", lw=2, label="generated vs CEP")
ax.set_xlabel("max Tanimoto similarity to training set"); ax.set_ylabel("density"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"生成分子: 中位数 {np.median(gen_sim):.3f}, 有 {np.mean(gen_sim > 0.9):.1%} 的相似度 > 0.9")
print(f"训练分子: 中位数 {np.median(inside_sim):.3f}, 有 {np.mean(inside_sim > 0.9):.1%} 的相似度 > 0.9")

📘 两条分布几乎重合：生成分子到训练集的最大相似度中位数约 **0.53**，训练分子彼此之间约 **0.50**。而且相似度 > 0.9 的生成分子只占 **0.2%**。

结论：**模型不是在背诵。** 它生成的分子和训练数据的"距离"，跟训练分子彼此之间的距离是一个量级——就像同一批化学家画出来的另一批分子。

💡 这个检验很重要。有些生成模型（特别是训练数据少、训练轮数多的时候）会出现大量"只改了一个原子"的近似复制品，novel 指标看起来很高（因为规范 SMILES 不同），但实际上毫无新意。只有画出相似度分布才能发现。

### 08a-3

In [ ]:
@torch.no_grad()
def greedy_sample(model, n=10):
    model.eval()
    x = torch.full((n, 1), stoi["^"]); state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    for _ in range(max_len):
        logits, state = model(x, state)
        x = logits[:, -1].argmax(dim=-1, keepdim=True)     # 取最大，不抽签
        out = torch.cat([out, x], dim=1)
        if (x[:, 0] == stoi["$"]).all():
            break
    result = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        result.append("".join(tokens))
    return result

greedy = greedy_sample(rnn, 10)
for s in greedy:
    print(s)
print("\n10 次生成里有", len(set(greedy)), "个不同的分子")

📘 10 次生成得到的是**同一个分子**，一字不差。

原因很简单：贪心解码里没有任何随机性。模型从 `^` 出发，每一步都取概率最大的 token，这个过程完全确定，所以无论跑多少次都走同一条路。

这就是 08a 第 6 节开头那个问题的答案：**生成模型的价值在于多样性，所以必须按概率抽签，而不是取最大值。**

💡 语言模型里有很多介于两者之间的解码策略：top-k 采样（只在概率最高的 k 个 token 里抽）、nucleus/top-p 采样（只在累计概率达到 p 的那些 token 里抽）、beam search（保留多条候选路径）。它们都是在"质量"和"多样性"之间找平衡，和温度是一类工具。

---
## 09a

先把 09a 的准备工作做一遍。为了让两次训练公平比较，这里把训练过程写成一个函数，**两次都训练 20 轮**（所以基线的重构率会比 09a 正文的 40 轮低一些）。

In [ ]:
try:
    import selfies
except ImportError:
    %pip install -q selfies rdkit
    import selfies

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import selfies as sf
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)
print("selfies 版本:", sf.__version__)

In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 cep.csv 即可
DATA_DIR = "../data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv（如果要用预训练模型，把 cep_rnn.pt 也一起上传）")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print("读取成功！一共", len(df), "个分子")
df.head()

In [ ]:
selfies_list = [sf.encoder(s) for s in df["smiles"]]
token_lists = [list(sf.split_selfies(s)) for s in selfies_list]

alphabet = ["[nop]"] + sorted({t for ts in token_lists for t in ts})    # [nop] 是补齐符
stoi = {t: i for i, t in enumerate(alphabet)}
max_len = max(len(ts) for ts in token_lists)
print("字母表大小:", len(alphabet), " 最长:", max_len, " 平均长度:", np.mean([len(ts) for ts in token_lists]).round(1))
print(alphabet)

In [ ]:
indices = torch.tensor([[stoi[t] for t in ts] + [0] * (max_len - len(ts)) for ts in token_lists])
X = nn.functional.one_hot(indices, len(alphabet)).float().reshape(len(indices), -1)
print("输入矩阵:", tuple(X.shape), f"（{max_len} 个位置 × {len(alphabet)} 种 token）")

perm = np.random.RandomState(0).permutation(len(X))
test_idx, train_idx = perm[:3000], perm[3000:]
print("训练:", len(train_idx), " 测试:", len(test_idx))

In [ ]:
LATENT_DIM = 128                                   # ✏️ 潜空间维数
INPUT_DIM = X.shape[1]

class MolecularVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)          # 分布的均值
        self.fc_logvar = nn.Linear(256, latent_dim)      # 分布的对数方差
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)      # 重参数化技巧
        return self.decoder(z), mu, logvar

vae = MolecularVAE(INPUT_DIM, LATENT_DIM)
print("参数量:", sum(p.numel() for p in vae.parameters()))

In [ ]:
def decode_latent(vae, z):
    with torch.no_grad():
        logits = vae.decoder(z)
    token_ids = logits.reshape(len(z), max_len, len(alphabet)).argmax(dim=-1)
    return [sf.decoder("".join(alphabet[i] for i in row if i != 0)) for row in token_ids.tolist()]

def canonical(smiles):
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return Chem.MolToSmiles(mol) if mol is not None and mol.GetNumAtoms() > 0 else None

def train_vae(beta, n_epochs=20, latent_dim=128, seed=0):
    torch.manual_seed(seed)
    vae = MolecularVAE(X.shape[1], latent_dim)
    optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
    rec_last = kl_last = 0.0
    for epoch in range(n_epochs):
        order = np.random.permutation(train_idx)
        rec_total = kl_total = 0.0
        warm = beta * min(1.0, (epoch + 1) / max(1, n_epochs // 3))
        for i in range(0, len(order), 256):
            xb = X[order[i:i + 256]]
            out, mu, logvar = vae(xb)
            rec = nn.functional.cross_entropy(
                out.reshape(-1, len(alphabet)), xb.reshape(-1, len(alphabet)).argmax(-1), reduction="sum") / len(xb)
            kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / len(xb)
            loss = rec + warm * kl
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            rec_total += rec.item(); kl_total += kl.item()
        n_batches = len(order) / 256
        rec_last, kl_last = rec_total / n_batches, kl_total / n_batches
    vae.eval()
    return vae, rec_last, kl_last

def vae_report(vae, name):
    with torch.no_grad():
        mu_test, _ = vae.encode(X[test_idx])
    recon = [canonical(s) for s in decode_latent(vae, mu_test[:1000])]
    originals = [canonical(s) for s in df["smiles"].values[test_idx[:1000]]]
    exact = np.mean([a is not None and a == b for a, b in zip(recon, originals)])
    print(f"{name}: 重构完全还原 {exact:.3f}   μ 每维标准差 {mu_test.std(0).mean():.3f}   "
          f"μ 到原点距离中位数 {mu_test.norm(dim=1).median():.2f}")
    return mu_test, exact

### 09a-1

In [ ]:
results = {}
for beta in [0.05, 0.5]:
    vae_b, rec, kl = train_vae(beta, n_epochs=20)
    mu_b, exact = vae_report(vae_b, f"beta = {beta}")
    results[f"beta = {beta}"] = {"reconstruction loss": round(rec, 2), "KL": round(kl, 2),
                                 "exact reconstruction": round(float(exact), 3),
                                 "mu std": round(float(mu_b.std(0).mean()), 3)}
    if beta == 0.05:
        vae, mu_test = vae_b, mu_b             # 后面两题用 beta=0.05 这个
pd.DataFrame(results)

📘 结果非常清楚地展示了**重构—规整权衡**：

| | β = 0.05 | β = 0.5 |
|---|---|---|
| 重构损失 | 9.5 | 22.9 |
| KL | 91.4 | 23.5 |
| 完全还原 | 2.7% | 0.1% |
| μ 每维标准差 | 0.65 | 0.20 |

β 大 10 倍以后：

1. **潜空间更"规整"了**：KL 从 91 降到 23；
2. **但重构全毁了**：完全还原率从 2.7% 掉到 0.1%。

🤔 等一下，β = 0.5 的 μ 标准差是 **0.20**，比 β = 0.05 的 0.65 **更远离**先验要求的 1.0。KL 明明变小了，怎么反而更不像标准正态分布？

📘 这正是**后验坍缩**（posterior collapse）的样子。KL 惩罚太重时，模型发现最省事的办法是让编码器**干脆不编码信息**：把所有分子的 μ 都压到原点附近（标准差趋近 0），同时让 σ 趋近 1。这样 KL 很小，但潜向量里没有任何关于分子的信息，解码器只能输出一个"平均分子"。

**KL 小不等于潜空间好用。** 要同时看重构。

💡 注意这里只训练了 20 轮，所以 β = 0.05 的完全还原率（2.7%）比正文 40 轮的（约 18%）低很多——重构率对训练轮数非常敏感。

### 09a-2

In [ ]:
from rdkit.Chem import Descriptors

z_prior = torch.randn(1000, 128)
prior_valid = [c for c in (canonical(s) for s in decode_latent(vae, z_prior)) if c]
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}
perfect = df["smiles"].sample(1000, random_state=0).tolist()     # "完美生成模型"：直接抄训练集

def describe(smiles_list, name, valid_rate):
    parsed = [Chem.MolFromSmiles(s) for s in smiles_list]
    unique = sorted({Chem.MolToSmiles(m) for m in parsed})
    return {
        "name": name,
        "valid": valid_rate,
        "unique": round(len(unique) / len(parsed), 3),
        "novel": round(len(set(unique) - canonical_train) / len(unique), 3),
        "heavy atoms": round(float(np.mean([m.GetNumHeavyAtoms() for m in parsed])), 1),
        "aromatic rings": round(float(np.mean([Descriptors.NumAromaticRings(m) for m in parsed])), 2),
        "fraction sp3 C": round(float(np.mean([Descriptors.FractionCSP3(m) for m in parsed])), 3),
    }

pd.DataFrame([describe(perfect, "训练集样本（完美模型）", 1.0),
              describe(prior_valid, "VAE 先验采样", len(prior_valid) / 1000)]).set_index("name")

📘 把"直接抄训练集"当作一个完美的生成模型，结果很有意思：

| | 训练集样本 | VAE 先验采样 |
|---|---|---|
| valid | 1.000 | 0.947 |
| unique | 1.000 | 0.992 |
| **novel** | **0.000** | **1.000** |
| 重原子数 | 27.5 | 21.9 |
| 芳香环数 | **5.71** | **0.41** |
| sp³ 碳比例 | 0.021 | 0.238 |

🤔 看 novel 这一行：完美模型得 **0 分**，VAE 得**满分**。

📘 如果只看 valid / unique / novel 这三个指标，**"直接抄训练集"会被判定为一个很差的生成模型，而生成大环和三元环的 VAE 会被判定为优秀**。

这显然荒谬，但它揭示了一件很实际的事：这三个指标只衡量"**不同**"，完全不衡量"**好**"。它们能排除最糟糕的失败（语法全错、模式崩溃、原样背诵），但**无法证明生成的分子有任何化学价值**。

真正要看的是：性质分布是否和训练数据一致（芳香环数 5.71 vs 0.41 已经说明一切），以及——最终——把分子拿去计算或实验。

### 09a-3 (a) 球面插值

In [ ]:
def slerp(a, b, t):
    a_n, b_n = a / a.norm(), b / b.norm()
    omega = torch.acos((a_n * b_n).sum().clamp(-1, 1))
    r = a.norm() + (b.norm() - a.norm()) * t            # 长度线性过渡
    return r * (torch.sin((1 - t) * omega) * a_n + torch.sin(t * omega) * b_n) / torch.sin(omega)

z_a, z_b = mu_test[0], mu_test[7]
ts = torch.linspace(0, 1, 8)
z_line = torch.stack([z_a + (z_b - z_a) * t for t in ts])
z_slerp = torch.stack([slerp(z_a, z_b, float(t)) for t in ts])

print("t      直线 ||z||   slerp ||z||")
for t, n1, n2 in zip(ts.tolist(), z_line.norm(dim=1).tolist(), z_slerp.norm(dim=1).tolist()):
    print(f"{t:.2f}      {n1:6.2f}      {n2:6.2f}")
print(f"\n真实分子 ||μ|| 中位数: {mu_test.norm(dim=1).median():.2f}")

for name, zs in [("直线插值", z_line), ("球面插值 slerp", z_slerp)]:
    smis = [canonical(s) for s in decode_latent(vae, zs)]
    rings = [Descriptors.NumAromaticRings(Chem.MolFromSmiles(s)) for s in smis if s]
    print(f"{name}: 路径上平均芳香环数 {np.mean(rings):.2f}")
    display(Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in smis if s], molsPerRow=8,
                                 subImgSize=(190, 160), legends=[f"t={t:.2f}" for t, s in zip(ts.tolist(), smis) if s]))

📘 先看 $\|z\|$：直线插值从 8.54 一路掉到 **6.18** 再回到 7.95，中间塌陷得很厉害；slerp 则稳定保持在 7.95–8.54 之间，和真实分子的中位数（约 9.3）接近。**slerp 确实修好了范数塌陷。**

🤔 那么分子质量变好了吗？

📘 **几乎没有。** 路径上的平均芳香环数：直线 1.62，slerp 1.38——不但没改善，还略差一点。

这个结果很值得琢磨。它说明：

1. 范数塌陷**只是问题之一**，不是全部。即使停留在"正确"的球壳上，两个特定分子之间的那条路径上仍然可能没有训练数据——高维空间太大了，球壳上也遍布空洞。
2. 更根本的瓶颈是**解码器太弱**（一个 MLP 一次性输出所有位置）。正文第 4 节已经看到，它连重构都只有 18%，自然也没能力在陌生区域生成合理的分子。

💡 **一个修好的诊断指标，不一定能修好模型。** 这是做研究时很常见的经验：先找到真正的瓶颈，再决定投入精力的方向。要让插值真正有用，得换更强的解码器（RNN/Transformer），而不是换插值方式。

### 09a-3 (b) 潜空间优化

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split as tts

pce_test = df["PCE"].values[test_idx]
Z = mu_test.numpy()
Z_tr, Z_te, p_tr, p_te = tts(Z, pce_test, test_size=0.3, random_state=0)
ridge = Ridge(alpha=1.0).fit(Z_tr, p_tr)
print(f"从 128 维潜向量预测 PCE: 测试 R² = {ridge.score(Z_te, p_te):.3f}")

start = int(np.argmax(pce_test))
direction = ridge.coef_ / np.linalg.norm(ridge.coef_)
steps = [0, 1, 2, 3, 5]
z_walk = torch.tensor(np.array([Z[start] + s * direction for s in steps]), dtype=torch.float32)
walk_smiles = [canonical(s) for s in decode_latent(vae, z_walk)]

print(f"起点分子真实 PCE: {pce_test[start]:.2f}%")
for s, smi in zip(steps, walk_smiles):
    pred = ridge.predict((Z[start] + s * direction).reshape(1, -1))[0]
    print(f"  step {s}: 岭回归预测 {pred:6.2f}%   {smi}")

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in walk_smiles if s], molsPerRow=5, subImgSize=(250, 190),
                     legends=[f"step {s}" for s, smi in zip(steps, walk_smiles) if smi])

📘 两个发现：

**1. 潜向量几乎预测不了 PCE：测试 R² 只有 0.08。** 这印证了正文第 5.2 节的判断——VAE 训练时没见过 PCE，潜空间是按"结构相似"组织的。

**2. 沿着岭回归的方向走，预测值一路飙到 66%。**

这是整个第五部分最该记住的一张表：

| 步数 | 岭回归预测的 PCE |
|---|---|
| 0 | 6.0% |
| 1 | 18.1% |
| 2 | 30.2% |
| 3 | 42.3% |
| 5 | **66.4%** |

⚠️ **数据集里 PCE 最高的分子是 11.09%**，而单结有机太阳能电池的理论极限大约在 20%–30%。66% 是物理上不可能的数字。

为什么会这样？因为**线性模型没有上界**：$\hat{y} = w \cdot z + b$，只要沿着 $w$ 的方向一直走，预测值就能无限增大。优化器忠实地执行了"让预测值最大"的指令，而这个指令本身毫无意义。

再看分子：走了几步之后出现了 `C=C=C` 这样的累积双键、`[SH]` 这样奇怪的基团，根本不是有机光伏材料。

💡 这就是 09b 第 4 节"模型欺骗"的完整演示，而且比随机森林更极端——随机森林至少**不会**预测出训练数据范围之外的值（见练习 09b-3），线性模型则毫无顾忌。**优化一个模型的输出之前，先问问这个模型在那个区域还成不成立。**

---
## 09b

⚠️ 上面 09a 部分把 `token_lists`、`alphabet`、`max_len`、`stoi` 这些名字换成了 **SELFIES** 的版本。
09b 要用回 **SMILES** 的词表，所以先把 08a 的那几格重新运行一遍，再做 09b 的准备工作。

In [ ]:
import copy
import random
from rdkit.Chem.rdchem import RWMol
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [ ]:
token_lists = [tokenize(s) for s in df["smiles"]]
counter = collections.Counter(t for ts in token_lists for t in ts)
print("一共", len(counter), "种 token")
pd.Series(counter).sort_values(ascending=False).to_frame("count")

In [ ]:
vocab = ["<pad>", "^", "$"] + sorted(counter)
stoi = {t: i for i, t in enumerate(vocab)}            # token -> 编号
max_len = max(len(ts) for ts in token_lists) + 2      # +2 是给 ^ 和 $ 留位置
print("词表大小:", len(vocab), " 最长序列:", max_len)
print(vocab)

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, :len(ids)] = torch.tensor(ids)

print("\n第一个分子:", df["smiles"][0])
print("编码后（0 是补齐）:", data[0][:20].tolist(), "...")

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = np.array([fp_gen.GetFingerprintAsNumPy(m) for m in mols], dtype=np.uint8)
y = df["PCE"].values

train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(fingerprints[train_idx], y[train_idx])

pred_test = rf.predict(fingerprints[test_idx])
rmse = np.sqrt(np.mean((pred_test - y[test_idx]) ** 2))
r2 = 1 - np.sum((pred_test - y[test_idx]) ** 2) / np.sum((y[test_idx] - y[test_idx].mean()) ** 2)
print(f"随机森林代理模型: 测试 RMSE = {rmse:.2f}%,  R² = {r2:.2f}")

In [ ]:
CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
if not os.path.exists(CKPT):
    raise FileNotFoundError("找不到 data/cep_rnn.pt。请先运行 05a，或把它和 cep.csv 一起上传。")

prior_model = SmilesRNN(len(vocab))                               # "先验模型"：05a 训练好的
prior_model.load_state_dict(torch.load(CKPT, map_location="cpu"))
prior_model.eval()
print("已加载预训练模型")

In [ ]:
top_fraction = 0.10                                               # ✏️ 试试 0.02 或 0.3
n_top = int(len(df) * top_fraction)
top_idx = np.argsort(-y)[:n_top]
print(f"微调集: {n_top} 个分子, PCE 门槛 {y[top_idx].min():.2f}%, 平均 {y[top_idx].mean():.2f}%")

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def analyse(model, name, n=2000, temperature=1.0):
    samples = sample(model, n, temperature=temperature)
    parsed = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in parsed if m is not None and m.GetNumAtoms() > 0]
    unique = sorted(set(valid))
    fps = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(c)) for c in unique], dtype=np.uint8)
    predictions = rf.predict(fps)
    print(f"{name:10s} valid {len(valid) / n:.3f}  unique {len(unique) / max(len(valid), 1):.3f}  "
          f"novel {len(set(unique) - canonical_train) / max(len(unique), 1):.3f}  "
          f"预测 PCE 平均 {predictions.mean():.2f}%  最大 {predictions.max():.2f}%")
    return pd.DataFrame({"smiles": unique, "pred_PCE": predictions})

### 09b-1

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

def finetune(fraction, n_epochs=4, lr=1e-4):
    model = copy.deepcopy(prior_model)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    rows = np.argsort(-y)[:int(len(df) * fraction)]
    subset = data[rows]
    model.train()
    for _ in range(n_epochs):
        order = np.random.permutation(len(subset))
        for i in range(0, len(order), 128):
            batch = subset[order[i:i + 128]]
            logits, _ = model(batch[:, :-1])
            loss = loss_fn(logits.reshape(-1, len(vocab)), batch[:, 1:].reshape(-1))
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5); optimizer.step()
    model.eval()
    return model, len(rows)

rows = []
for fraction in [0.01, 0.10, 0.30]:
    model_f, n_used = finetune(fraction)
    samples = sample(model_f, 1000)
    parsed = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in parsed if m is not None and m.GetNumAtoms() > 0]
    unique = sorted(set(valid))
    fps = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(c)) for c in unique], dtype=np.uint8)
    rows.append({
        "fine-tune set": f"top {fraction:.0%} ({n_used})",
        "valid": len(valid) / 1000,
        "unique": round(len(unique) / max(len(valid), 1), 3),
        "novel": round(len(set(unique) - canonical_train) / max(len(unique), 1), 3),
        "mean predicted PCE": round(float(rf.predict(fps).mean()), 2),
    })
pd.DataFrame(rows).set_index("fine-tune set")

📘 结果和"微调集越小越容易背书"的直觉**不完全一样**：

| 微调集 | unique | novel | 平均预测 PCE |
|---|---|---|---|
| top 1%（299 个） | 0.999 | 0.985 | 4.75% |
| top 10%（2997 个） | 0.998 | 0.984 | **5.90%** |
| top 30%（8993 个） | 1.000 | 0.983 | 5.61% |

两个观察：

1. **唯一性和新颖性几乎没变**，都在 0.98 以上。预想中的"模式崩溃"没有出现——因为只微调了 4 轮、学习率只有 1e-4，模型还没来得及把那 299 个分子背下来。
2. **平均预测 PCE 在 top 10% 时最高**，top 1% 反而更低。

🤔 为什么用"最好的 1%"反而效果更差？

📘 299 个分子提供的**信号太少**：模型从中学到的更多是这几百个分子的偶然特征，而不是"高 PCE 分子的共性"。这和第二部分训练集太小时模型学不好是同一回事——**微调也需要足够的数据量**。

另一头，top 30% 的门槛太低（包含了大量中等分子），分布推不动。10% 在"数据够多"和"目标够高"之间取得了平衡。

💡 如果真要用 1% 这样的小集合，就得更小心：更低的学习率、更少的轮数，并且密切监控 unique 指标。继续微调下去（比如 20 轮），模式崩溃就会开始出现。

### 09b-2

In [ ]:
FRAGMENTS = ["c1ccsc1", "c1ccoc1", "c1cc[nH]c1", "c1ccccc1",
             "c1cc[se]c1", "c1ccc2nsnc2c1", "c1ccc2[nH]ccc2c1"]

def parse_chain(chain):
    mol = Chem.MolFromSmiles("-".join(chain))
    return mol if mol is not None and mol.GetNumAtoms() > 0 else None

def tree_predictions(mol):
    x = np.array([fp_gen.GetFingerprintAsNumPy(mol)], dtype=np.uint8)
    return np.array([t.predict(x)[0] for t in rf.estimators_])

def similarity_to_train(mol):
    return max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), reference_fps))

def propose(chain, rng):
    new = list(chain)
    move = rng.random()
    if move < 0.45 and len(new) < 20:
        new.insert(rng.randrange(len(new) + 1), rng.choice(FRAGMENTS))
    elif move < 0.65 and len(new) > 2:
        new.pop(rng.randrange(len(new)))
    else:
        new[rng.randrange(len(new))] = rng.choice(FRAGMENTS)
    return new

def hill_climb(constrained, n_steps=150, seed=0):
    rng = random.Random(seed)
    def score(mol):
        preds = tree_predictions(mol)
        return preds.mean() - 2 * preds.std() if constrained else preds.mean()
    chain = [rng.choice(FRAGMENTS) for _ in range(3)]
    best_mol = parse_chain(chain)
    best_score = score(best_mol)
    for _ in range(n_steps):
        candidate = propose(chain, rng)
        mol = parse_chain(candidate)
        if mol is None:
            continue
        if constrained and similarity_to_train(mol) < 0.4:   # 跑出适用域就拒绝
            continue
        s = score(mol)
        if s > best_score:
            chain, best_mol, best_score = candidate, mol, s
    return chain, best_mol

results_hc = {}
for constrained in [False, True]:
    label = "有约束" if constrained else "无约束"
    chain, mol = hill_climb(constrained)
    preds = tree_predictions(mol)
    results_hc[label] = mol
    print(f"{label}: 预测 PCE {preds.mean():.2f}%  悲观分 {preds.mean() - 2 * preds.std():.2f}%  "
          f"相似度 {similarity_to_train(mol):.3f}  树的分歧 {preds.std():.2f}%  "
          f"重原子 {mol.GetNumHeavyAtoms()}  片段数 {len(chain)}")
    print("   ", "-".join(chain))

Draw.MolsToGridImage(list(results_hc.values()), molsPerRow=2, subImgSize=(330, 250), legends=list(results_hc))

📘 两个版本的终点：

| | 无约束 | 有约束 |
|---|---|---|
| 预测 PCE | **6.45%** | 5.88% |
| 悲观分（减 2σ） | 3.92% | **4.11%** |
| 到训练集的相似度 | **0.394** | 0.439 |
| 树的分歧 | **1.26%** | 0.88% |
| 重原子数 | 42 | 30 |
| 片段数 | 6 | 5 |

无约束的版本确实拿到了更高的**名义**分数（6.45% vs 5.88%），但代价是：

- 相似度 0.394，低于训练集内部的中位数（约 0.50），已经在适用域边缘；
- 树的分歧 1.26%，高于测试集的平均水平（约 1.07%）——模型自己在说"没把握"；
- 分子长到了 42 个重原子，超出了训练数据 13–35 的范围。

**按悲观估计排序，结论反过来了**：有约束版本的 4.11% 高于无约束版本的 3.92%。

🤔 哪个更值得拿去做 DFT？

📘 **有约束的那个。** 理由不是它的预测值更高，而是这个预测值**更可信**：分子在适用域内、大小正常、各棵树看法一致。

拿一个名义上 6.45% 但模型没把握的分子去做几小时的 DFT，期望收益反而更低。**在真实的材料发现里，计算和实验资源有限，"可信的中等预测"常常比"不可信的高预测"更有价值。**

💡 这正是**贝叶斯优化**要解决的问题：在"利用"（选预测值高的）和"探索"（选不确定度高的）之间做有原则的权衡，而不是简单地取最大值。ai4chem 第 10 章讲的就是这个。

### 09b-3

In [ ]:
low_mask = y < 6
high_mask = y > 8
print(f"只用 PCE < 6% 的 {low_mask.sum()} 个分子训练，预测 PCE > 8% 的 {high_mask.sum()} 个分子")

rf_low = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_low.fit(fingerprints[low_mask], y[low_mask])
pred_high = rf_low.predict(fingerprints[high_mask])

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(y[high_mask], pred_high, s=4, alpha=0.3)
ax.plot([5, 12], [5, 12], "k--", lw=1)
ax.axhline(6, color="tab:red", ls=":", label="training data ceiling (6%)")
ax.set_xlabel("true DFT PCE (%)"); ax.set_ylabel("predicted PCE (%)"); ax.legend(fontsize=8)
ax.set_title("Can the model find molecules better than it has seen?")
plt.tight_layout(); plt.show()

print(f"真实平均 {y[high_mask].mean():.2f}%,  预测平均 {pred_high.mean():.2f}%,  "
      f"平均低估 {y[high_mask].mean() - pred_high.mean():.2f}%")
print(f"预测值超过 6%（训练数据上限）的比例: {np.mean(pred_high > 6):.1%}")

📘 结果非常干脆：

- 真实 PCE 平均 **9.16%**，模型预测平均只有 **2.47%**，平均低估 **6.69 个百分点**；
- 预测值超过 6%（训练数据的上限）的比例：**0.0%**。一个都没有。

**随机森林完全没有能力发现比它见过的更好的分子。**

🤔 为什么会这样？

📘 随机森林的预测是**叶子节点里训练样本 y 的平均值**。既然所有训练样本的 PCE 都 < 6%，任何叶子的平均值也必然 < 6%，所以**预测值在数学上不可能超过 6%**。

这和 02a 里随机森林把并五苯预测得和并四苯一样是**完全相同**的机制：基于树的模型无法外推。

⚠️ 这对定向生成意味着什么？

> 如果你用一个代理模型去筛选"比现有材料更好"的候选分子，而这个模型**在结构上就不可能给出超出训练范围的预测**，那么你永远只能找到"和已知最好的差不多"的分子。

这是生成 + 筛选流程里一个很隐蔽的天花板。应对办法：

1. 用**能外推**的模型（线性模型、高斯过程、神经网络）——但要记住练习 09a-3(b) 的教训，能外推也意味着能胡说；
2. 用**排序**而不是绝对值：模型区分"更好/更差"的能力通常比预测准确数值的能力强；
3. **主动学习闭环**：把筛出来的分子真的拿去算 DFT，把结果加回训练集。只有这样，模型的"视野上限"才能真正抬高。